In [ ]:
!pip install langchain langchain_openai pymupdf langchain_community faiss-cpu

In [ ]:
#import the necessary libraries
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import fitz
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
import base64
from dotenv import load_dotenv
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

load_dotenv()
import os

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL=os.getenv("OPENAI_BASE_URL")
FAST_MODEL_NAME=os.getenv("FAST_MODEL_NAME")

In [ ]:
# loader = PyMuPDFLoader(
#     file_path="../Shivam Kumar Thakur - Offer Letter & Annexure A & B.pdf",
#     extract_images = True,
#     extract_tables="markdown",
#     mode="page",
    
# )

# loaded_documents = loader.load()

doc = fitz.open("../Shivam Kumar Thakur - Offer Letter & Annexure A & B.pdf")
len(doc)  # number of pages in the PDF

In [ ]:
#extract images, tables, and text from the loaded documents
images = []
tables = []
texts = []
for i, page in enumerate(doc):
    #process text
    text = page.get_text()
    if text.strip():
        texts.append({
            "page_index": i,
            "text": text
        })

    #process images
    image_list = page.get_images(full=True)
    for img_index, img in enumerate(image_list):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        base64_image = base64.b64encode(image_bytes).decode("utf-8")
        images.append({
            "page_index": i,
            "image_index": img_index,
            "image_base64": base64_image
        })    
    
    #process tables
    table_list = page.find_tables()
    for table_index, table in enumerate(table_list):
        df  = table.to_pandas()
        tables.append({
            "page_index": i,
            "table_index": table_index,
            "table_df": df
        })

# texts, images, tables        
len(images)

In [ ]:
# Define prompt templates and LLMs for batch caption generation
from langchain_core.prompts import PromptTemplate
import time
from datetime import datetime

# Image caption prompt template
image_caption_prompt = PromptTemplate(
    input_variables=["image_description"],
    template="""
You are creating metadata for a Retrieval-Augmented Generation (RAG) system.

Analyze the following image and generate a detailed description that will maximize semantic search quality.

Your description should include:
- The type of image (photo, diagram, chart, graph, flowchart, screenshot, logo, etc.)
- All visible objects and entities
- Important labels, titles, headings, legends, axes, and annotations
- Relationships between objects
- Any numbers, measurements, dates, percentages, or statistics
- Any text appearing in the image (OCR)
- The overall purpose or meaning of the image
- Important keywords and technical terminology

Write as one detailed paragraph.
Do not mention image quality or colors unless they are important.

Image:
data:image/png;base64,{image_description}
"""
)

# Table caption prompt template
table_caption_prompt = PromptTemplate(
    input_variables=["table_description"],
    template="""
You are creating metadata for a Retrieval-Augmented Generation (RAG) system.

Analyze the following table and produce a retrieval-friendly description.

Include:
- What the table represents
- Column names
- Row categories
- Key values and trends
- Highest and lowest values
- Important comparisons
- Units, dates, percentages, currencies if present
- Important keywords
- A concise summary of the information

Return one detailed paragraph suitable for semantic embedding.

Table:
{table_description}
"""
)

# Create LLM instances reused for batch processing
image_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)
table_llm = ChatOpenAI(
    model_name=FAST_MODEL_NAME,
    openai_api_key=OPENAI_API_KEY,
    openai_api_base=OPENAI_BASE_URL,

    temperature=0.3)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

documents = []

# Start timing
start_time = time.time()
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

for _text in texts:
    page_index = _text["page_index"]
    text = _text["text"]
    chunks = text_splitter.split_text(text)
    for chunk in chunks:
        documents.append({
            "page_index": page_index,
            "text_chunk": chunk,
            "type":"text"
        })

print("texts processed., total documents:", len(documents))

# Batch process images — parallel caption generation
if images:
    t0 = time.time()
    image_prompts = [image_caption_prompt.format(image_description=img["image_base64"]) for img in images]
    image_captions = image_llm.batch(image_prompts)
    for img, caption in zip(images, image_captions):
        documents.append({
            "page_index": img["page_index"],
            "image_caption": caption.content if hasattr(caption, 'content') else caption,
            "type": "image",
            "image_index": img["image_index"]
        })
    print(f"Images processed in {time.time() - t0:.2f}s, total documents: {len(documents)}")

# Batch process tables — parallel caption generation
if tables:
    t0 = time.time()
    table_prompts = [table_caption_prompt.format(table_description=t["table_df"].to_string()) for t in tables]
    table_captions = table_llm.batch(table_prompts)
    for t, caption in zip(tables, table_captions):
        documents.append({
            "page_index": t["page_index"],
            "table_caption": caption.content if hasattr(caption, 'content') else caption,
            "type": "table",
            "table_index": t["table_index"]
        })
    print(f"Tables processed in {time.time() - t0:.2f}s, total documents: {len(documents)}")

end_time = time.time()
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total processing time: {end_time - start_time:.2f}s")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_core.documents import Document
import os

# Create embeddings model
embeddings = HuggingFaceEndpointEmbeddings(
            repo_id="BAAI/bge-base-en-v1.5",
            huggingfacehub_api_token=os.getenv("HUGGING_FACE_TOKEN")
        )
# Convert documents list to LangChain Document objects with metadata
docs = []
for item in documents:
    if item["type"] == "text":
        page_content = item["text_chunk"]
        metadata = {
            "type": "text",
            "page_index": item["page_index"]
        }
    elif item["type"] == "image":
        page_content = item["image_caption"]
        metadata = {
            "type": "image",
            "page_index": item["page_index"],
            "image_index": item["image_index"]
        }
    elif item["type"] == "table":
        page_content = item["table_caption"]
        metadata = {
            "type": "table",
            "page_index": item["page_index"],
            "table_index": item["table_index"]
        }
    
    doc = Document(page_content=page_content, metadata=metadata)
    docs.append(doc)

print(f"Total documents to index: {len(docs)}")

# Create FAISS vector store from documents
vector_store = FAISS.from_documents(docs, embeddings)

# Save the FAISS index locally
index_path = "faiss_index"
vector_store.save_local(index_path)

print(f"FAISS vector store created and saved to '{index_path}/'")
print(f"Index contains {vector_store.index.ntotal} vectors")


In [ ]:
retrieve_docs = vector_store.similarity_search("What is the salary mentioned in the offer letter?", k=3)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant that answers questions based on the provided context.
Context:
{context}

input: {input}
    """
)

def format_doc(doc):
    return "\n\n".join(doc.page_content for doc in docs)

runnable = RunnableParallel({
    "input": RunnablePassthrough(),
    "context": retriever | format_doc
})


chain = runnable | prompt | table_llm

In [ ]:
chain.invoke("What is the salary mentioned in the offer letter?")

